In [1]:
import os
import argparse
import random
import time
import warnings
warnings.filterwarnings("ignore")
 
import h5py
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
 
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision.transforms.functional import rotate as tvrotate
from torchvision import transforms
from tqdm.auto import tqdm

def set_seed(s: int = 42):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
 
set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
def get_args():
    p = argparse.ArgumentParser(description="Bonus Task – Exp 1: Rotation Baselines")
    p.add_argument("--task1_ckpt",  type=str,
                   default="/kaggle/input/models/vitthal2945/e2e-vaemodel/pytorch/default/1/best_model.pt",
                   help="Path to Task-1 VAE checkpoint (best_model.pt)")
    p.add_argument("--data_dir",    type=str,
                   default="/kaggle/input/datasets/vitthal2945/e2e-hidden-symmetries-dataset",
                   help="Directory containing rotated_mnist_train.h5 / test.h5")
    p.add_argument("--out_dir",     type=str,
                   default="./bonus_outputs/exp1")
    p.add_argument("--latent_dim",  type=int,  default=16)
    p.add_argument("--epochs",      type=int,  default=30)
    p.add_argument("--batch_size",  type=int,  default=256)
    p.add_argument("--lr",          type=float, default=1e-3)
    p.add_argument("--n_bootstrap", type=int,  default=500,
                   help="Bootstrap iterations for 95% CI on per-angle accuracy")
    return p.parse_args([])

In [3]:
class ResBlock(nn.Module):
    def __init__(self, ch, g=8):
        super().__init__()
        g = min(g, ch)
        self.net = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1, bias=False), nn.GroupNorm(g, ch), nn.SiLU(),
            nn.Conv2d(ch, ch, 3, padding=1, bias=False), nn.GroupNorm(g, ch),
        )
        self.act = nn.SiLU()
    def forward(self, x): return self.act(x + self.net(x))
 
class Encoder(nn.Module):
    def __init__(self, ld):
        super().__init__()
        self.stem  = nn.Sequential(nn.Conv2d(1, 32, 4, 2, 1, bias=False), nn.GroupNorm(8, 32), nn.SiLU())
        self.l1    = nn.Sequential(ResBlock(32),  nn.Conv2d(32,  64, 4, 2, 1, bias=False), nn.GroupNorm(8,  64), nn.SiLU())
        self.l2    = nn.Sequential(ResBlock(64),  nn.Conv2d(64, 128, 3, 2, 1, bias=False), nn.GroupNorm(8, 128), nn.SiLU())
        self.l3    = ResBlock(128)
        self.fc    = nn.Sequential(nn.Flatten(), nn.Linear(128*4*4, 512), nn.SiLU())
        self.fc_mu = nn.Linear(512, ld)
        self.fc_lv = nn.Linear(512, ld)
    def forward(self, x):
        h = self.l3(self.l2(self.l1(self.stem(x))))
        h = self.fc(h)
        return self.fc_mu(h), self.fc_lv(h)
 
class Decoder(nn.Module):
    def __init__(self, ld):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(ld, 512), nn.SiLU(), nn.Linear(512, 128*4*4), nn.SiLU())
        self.u1 = nn.Sequential(ResBlock(128), nn.ConvTranspose2d(128, 64, 3, 2, 1, output_padding=1), nn.GroupNorm(8,  64), nn.SiLU())
        self.u2 = nn.Sequential(ResBlock(64),  nn.ConvTranspose2d(64,  32, 4, 2, 1),                  nn.GroupNorm(8,  32), nn.SiLU())
        self.u3 = nn.Sequential(ResBlock(32),  nn.ConvTranspose2d(32,   1, 4, 2, 3),                  nn.Sigmoid())
    def forward(self, z): return self.u3(self.u2(self.u1(self.fc(z).view(-1, 128, 4, 4))))
 
class Task1VAE(nn.Module):
    def __init__(self, ld):
        super().__init__()
        self.encoder = Encoder(ld)
        self.decoder = Decoder(ld)
    def encode(self, x):
        mu, _ = self.encoder(x)
        return mu
 
def load_vae(ckpt_path: str, latent_dim: int) -> Task1VAE:
    vae = Task1VAE(latent_dim).to(device)
    if os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location=device, weights_only=False)
        # Handle both wrapped and unwrapped state dicts
        sd = ck.get("model", ck)
        vae.load_state_dict(sd)
        print(f"  ✓ Task-1 VAE loaded  (epoch {ck.get('epoch', '?')})")
    else:
        print(f"  ⚠  Task-1 checkpoint not found at {ckpt_path} – using random weights")
    vae.eval()
    for p in vae.parameters():
        p.requires_grad = False
    return vae

In [4]:
def load_h5(path: str):
    print(f"  loading {os.path.basename(path)} ...", end=" ", flush=True)
    t0 = time.time()
    with h5py.File(path, "r") as f:
        imgs   = torch.from_numpy(f["images"][:].astype("float32")).unsqueeze(1).clamp(0., 1.)
        labels = torch.from_numpy(f["labels"][:].astype("int64"))
        angles = torch.from_numpy(f["angles"][:].astype("int64"))
    print(f"done ({time.time()-t0:.1f}s)  shape={tuple(imgs.shape)}")
    return imgs, labels, angles
 
@torch.no_grad()
def pre_encode(vae: Task1VAE, imgs: torch.Tensor, batch: int = 512) -> torch.Tensor:
    """Encode entire split; returns CPU tensor of mu vectors."""
    Z = []
    for i in tqdm(range(0, len(imgs), batch), desc="  encoding", leave=False):
        Z.append(vae.encode(imgs[i:i+batch].to(device)).cpu())
    return torch.cat(Z)
 
@torch.no_grad()
def pre_encode_all_angles(
    vae: Task1VAE,
    imgs_0deg: torch.Tensor,
    angles: list,
    batch: int = 512
) -> dict:
    """
    For evaluation only.
    Returns dict: angle → latent tensor (same samples, rotated on-the-fly).
    """
    result = {}
    for angle in angles:
        Z = []
        for i in range(0, len(imgs_0deg), batch):
            chunk = imgs_0deg[i:i+batch]
            if angle != 0:
                chunk = tvrotate(chunk, float(angle),
                                 interpolation=transforms.InterpolationMode.BILINEAR,
                                 fill=[0.])
            Z.append(vae.encode(chunk.to(device)).cpu())
        result[angle] = torch.cat(Z)
    return result

In [5]:
class MLPClassifier(nn.Module):
    """
    3-layer MLP operating on latent codes.
    Architecture matches Task-3 Oracle for fair comparison.
    """
    def __init__(self, ld: int, nc: int, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(ld, hidden), nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, nc),
        )
    def forward(self, z): return self.net(z)
 
def train_classifier(
    Z_train: torch.Tensor,
    L_train: torch.Tensor,
    latent_dim: int,
    n_classes: int,
    epochs: int,
    batch_size: int,
    lr: float,
    tag: str = ""
) -> tuple:
    """
    Train MLP classifier on (Z_train, L_train).
    Returns (trained model, history dict).
    """
    clf   = MLPClassifier(latent_dim, n_classes).to(device)
    opt   = torch.optim.Adam(clf.parameters(), lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loader = DataLoader(TensorDataset(Z_train, L_train),
                        batch_size=batch_size, shuffle=True)
    history = {"train_acc": [], "train_loss": []}
    print(f"\n  Training {tag} for {epochs} epochs …")
    for ep in range(1, epochs + 1):
        clf.train()
        correct = total = loss_sum = 0
        for z, lbl in loader:
            z, lbl = z.to(device), lbl.to(device)
            opt.zero_grad()
            logits = clf(z)
            loss = F.cross_entropy(logits, lbl)
            loss.backward()
            opt.step()
            correct   += (logits.detach().argmax(1) == lbl).sum().item()
            total     += z.size(0)
            loss_sum  += loss.item() * z.size(0)
        sched.step()
        history["train_acc"].append(correct / total)
        history["train_loss"].append(loss_sum / total)
        if ep % 10 == 0 or ep == 1:
            print(f"    ep {ep:3d}/{epochs}  acc={correct/total:.4f}  loss={loss_sum/total:.4f}")
    clf.eval()
    return clf, history

In [6]:
@torch.no_grad()
def per_angle_accuracy(
    clf: MLPClassifier,
    Z_by_angle: dict,
    labels: torch.Tensor,
    batch: int = 512
) -> dict:
    """Returns {angle: accuracy} using pre-encoded latents per angle."""
    results = {}
    for angle, Z in Z_by_angle.items():
        correct = total = 0
        for i in range(0, len(Z), batch):
            z_b = Z[i:i+batch].to(device)
            l_b = labels[i:i+batch].to(device)
            pred = clf(z_b).argmax(1)
            correct += (pred == l_b).sum().item()
            total   += z_b.size(0)
        results[angle] = correct / total
    return results
 
@torch.no_grad()
def bootstrap_ci(
    clf: MLPClassifier,
    Z: torch.Tensor,
    labels: torch.Tensor,
    n_bootstrap: int = 500,
    ci: float = 0.95
) -> tuple:
    """
    Bootstrap 95% CI for accuracy on a single (Z, labels) pair.
    Returns (mean, lower, upper).
    """
    n = len(Z)
    rng = np.random.default_rng(42)
    accs = []
    with torch.no_grad():
        logits_all = []
        for i in range(0, n, 512):
            logits_all.append(clf(Z[i:i+512].to(device)).cpu())
        logits_all = torch.cat(logits_all)
        preds = logits_all.argmax(1).numpy()
        true  = labels.numpy()
    correct = (preds == true).astype(float)
    for _ in range(n_bootstrap):
        idx  = rng.integers(0, n, size=n)
        accs.append(correct[idx].mean())
    alpha = (1 - ci) / 2
    return float(np.mean(accs)), float(np.percentile(accs, alpha*100)), float(np.percentile(accs, (1-alpha)*100))
 
@torch.no_grad()
def rotation_equivariance_error(
    clf: MLPClassifier,
    Z_by_angle: dict,
    batch: int = 256
) -> dict:
    """
    REE(θ) = mean_z ||softmax(f(z)) - softmax(f(z_θ))||_2
    Measures drift in the OUTPUT distribution (not just class label).
    Lower is better; 0 means perfectly invariant logits.
    """
    Z0 = Z_by_angle[0]
    results = {}
    for angle, Zth in Z_by_angle.items():
        if angle == 0:
            results[0] = 0.0
            continue
        diffs = []
        for i in range(0, len(Z0), batch):
            p0  = F.softmax(clf(Z0[i:i+batch].to(device)), dim=1)
            pth = F.softmax(clf(Zth[i:i+batch].to(device)), dim=1)
            diffs.extend((p0 - pth).norm(dim=1).cpu().numpy().tolist())
        results[angle] = float(np.mean(diffs))
    return results
 
@torch.no_grad()
def rotation_sensitivity_manifold(
    clf: MLPClassifier,
    Z0_sample: torch.Tensor,      # (N, ld) subset at 0°
    labels_sample: torch.Tensor,
    angles: list
) -> np.ndarray:
    """
    RSM[i, k] = entropy of softmax(f(z_i rotated by angles[k]))
    Shape: (N, len(angles))
    High entropy → classifier is uncertain → rotation broke the representation.
    """
    n = len(Z0_sample)
    K = len(angles)
    entropy_mat = np.zeros((n, K))
    base_imgs   = None   # we need original images for this — use stored mapping
    # We only have latents here, so we compute entropy of latent-space prediction
    for k, angle in enumerate(angles):
        # We apply a linear approximation: z_θ ≈ z_0 + θ·J·g(z_0)
        # But since we have pre-encoded latents per angle, use those
        # (This function is called with actual Z per angle via partial application)
        pass
    # Return entropy from the 0° latents only (full angle version in main)
    return entropy_mat
 
@torch.no_grad()
def confusion_angle_matrix(
    clf: MLPClassifier,
    Z_by_angle: dict,
    labels: torch.Tensor,
    n_classes: int
) -> np.ndarray:
    """
    CAM[c, k] = accuracy of class c at angle index k
    Reveals which digits are most rotation-sensitive.
    Shape: (n_classes, n_angles)
    """
    angles = sorted(Z_by_angle.keys())
    cam    = np.zeros((n_classes, len(angles)))
    with torch.no_grad():
        for k, angle in enumerate(angles):
            Z = Z_by_angle[angle]
            logits_all = torch.cat([clf(Z[i:i+512].to(device)).cpu()
                                    for i in range(0, len(Z), 512)])
            preds = logits_all.argmax(1)
            for c in range(n_classes):
                mask = labels == c
                if mask.sum() > 0:
                    cam[c, k] = (preds[mask] == labels[mask]).float().mean().item()
    return cam

In [7]:
def rotation_accuracy_gap(acc_by_angle: dict) -> float:
    """
    RAG = acc(0°) − mean(acc(θ≠0°))
    Quantifies how much accuracy is lost by rotation.
    Perfect invariance → RAG = 0.
    """
    acc0      = acc_by_angle[0]
    acc_other = [v for k, v in acc_by_angle.items() if k != 0]
    return acc0 - float(np.mean(acc_other))

In [8]:
DARK, PANEL = "#0d0d0d", "#1a1a2e"
COLORS      = {"noaug": "#e94560", "fullaug": "#0f9b8e"}
 
def _ax(ax, title="", xl="", yl=""):
    ax.set_facecolor(PANEL); ax.tick_params(colors="white")
    for s in ax.spines.values(): s.set_edgecolor("#444")
    if title: ax.set_title(title, color="white", fontsize=9)
    if xl:    ax.set_xlabel(xl, color="white", fontsize=8)
    if yl:    ax.set_ylabel(yl, color="white", fontsize=8)
 
def plot_accuracy_profile(
    acc_noaug: dict, acc_fullaug: dict,
    ci_noaug: dict, ci_fullaug: dict,
    ree_noaug: dict, ree_fullaug: dict,
    history_noaug: dict, history_fullaug: dict,
    out_path: str
):
    angles = sorted(acc_noaug.keys())
    x = np.array(angles)
 
    fig = plt.figure(figsize=(20, 12), facecolor=DARK)
    fig.suptitle("Exp 1 — Rotation Sensitivity Baselines\n"
                 "MLP-NoAug (trained 0° only)  vs  MLP-FullAug (trained all rotations)",
                 color="white", fontsize=12, fontweight="bold")
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.40, wspace=0.35)
 
    # Panel 1: Per-angle accuracy + CI
    ax1 = fig.add_subplot(gs[0, :2])
    _ax(ax1, "Per-Angle Classification Accuracy (± 95% Bootstrap CI)",
        "Rotation Angle θ (°)", "Accuracy")
    for key, acc, ci, col in [
        ("MLP-NoAug",   acc_noaug,  ci_noaug,  COLORS["noaug"]),
        ("MLP-FullAug", acc_fullaug, ci_fullaug, COLORS["fullaug"]),
    ]:
        y    = np.array([acc[a] for a in angles])
        lo   = np.array([acc[a] - ci[a][1] for a in angles])
        hi   = np.array([ci[a][2] - acc[a] for a in angles])
        ax1.plot(x, y, "o-", color=col, lw=2, label=key)
        ax1.fill_between(x, y - lo, y + hi, alpha=0.15, color=col)
    ax1.axhline(np.mean(list(acc_noaug.values())),   ls="--", lw=1, color=COLORS["noaug"],   alpha=0.6)
    ax1.axhline(np.mean(list(acc_fullaug.values())), ls="--", lw=1, color=COLORS["fullaug"], alpha=0.6)
    ax1.set_xticks(x); ax1.set_xticklabels([f"{a}°" for a in angles], color="white", fontsize=8)
    ax1.legend(facecolor=PANEL, labelcolor="white")
    ax1.set_ylim(0, 1.05)
 
    # Panel 2: RAG bar
    ax2 = fig.add_subplot(gs[0, 2])
    _ax(ax2, "Rotation Accuracy Gap (RAG)\nacc(0°) − mean(acc(θ≠0°))\nLower = more invariant", "", "RAG")
    rag_na = rotation_accuracy_gap(acc_noaug)
    rag_fa = rotation_accuracy_gap(acc_fullaug)
    bars = ax2.bar(["NoAug", "FullAug"], [rag_na, rag_fa],
                   color=[COLORS["noaug"], COLORS["fullaug"]], alpha=0.85, edgecolor=DARK)
    for bar, v in zip(bars, [rag_na, rag_fa]):
        ax2.text(bar.get_x() + bar.get_width()/2, v + 0.005,
                 f"{v:.4f}", ha="center", color="white", fontsize=10, fontweight="bold")
    ax2.set_facecolor(PANEL); ax2.tick_params(colors="white")
    for s in ax2.spines.values(): s.set_edgecolor("#444")
 
    # Panel 3: Rotation Equivariance Error
    ax3 = fig.add_subplot(gs[1, :2])
    _ax(ax3, "Rotation Equivariance Error in Logit Space\n"
        "REE(θ) = mean_z ||softmax(f(z)) − softmax(f(z_θ))||₂\n"
        "Measures drift in OUTPUT DISTRIBUTION (stronger than accuracy alone)",
        "Rotation Angle θ (°)", "REE")
    for key, ree, col in [
        ("MLP-NoAug",   ree_noaug,  COLORS["noaug"]),
        ("MLP-FullAug", ree_fullaug, COLORS["fullaug"]),
    ]:
        y_ree = np.array([ree[a] for a in angles])
        ax3.plot(x, y_ree, "s-", color=col, lw=2, label=key)
    ax3.set_xticks(x); ax3.set_xticklabels([f"{a}°" for a in angles], color="white", fontsize=8)
    ax3.legend(facecolor=PANEL, labelcolor="white")
 
    # Panel 4: Training curves
    ax4 = fig.add_subplot(gs[1, 2])
    _ax(ax4, "Training Accuracy Curves", "Epoch", "Train Acc")
    ax4.plot(history_noaug["train_acc"],  color=COLORS["noaug"],   lw=2, label="NoAug")
    ax4.plot(history_fullaug["train_acc"], color=COLORS["fullaug"], lw=2, label="FullAug")
    ax4.legend(facecolor=PANEL, labelcolor="white")
    ax4.set_ylim(0, 1.05)
 
    plt.tight_layout()
    plt.show()
    plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor=DARK)
    plt.close()
    print(f"  saved → {out_path}")
 
 
def plot_confusion_angle(
    cam_noaug: np.ndarray,
    cam_fullaug: np.ndarray,
    angles: list,
    digit_names: list,
    out_path: str
):
    n_classes = cam_noaug.shape[0]
    fig, axes = plt.subplots(1, 2, figsize=(18, 7), facecolor=DARK)
    fig.suptitle("Confusion-Angle Matrix: per-class accuracy at each rotation\n"
                 "Dark = low accuracy  |  Bright = high accuracy",
                 color="white", fontsize=11, fontweight="bold")
    for ax, cam, tag in [(axes[0], cam_noaug, "MLP-NoAug"), (axes[1], cam_fullaug, "MLP-FullAug")]:
        im = ax.imshow(cam, cmap="viridis", vmin=0, vmax=1, aspect="auto")
        ax.set_facecolor(PANEL)
        ax.set_xticks(range(len(angles)))
        ax.set_xticklabels([f"{a}°" for a in angles], color="white", fontsize=7, rotation=45)
        ax.set_yticks(range(n_classes))
        ax.set_yticklabels([str(d) for d in digit_names], color="white", fontsize=8)
        ax.set_xlabel("Rotation Angle", color="white", fontsize=9)
        ax.set_ylabel("Digit Class", color="white", fontsize=9)
        ax.set_title(tag, color="white", fontsize=10)
        for i in range(n_classes):
            for j in range(len(angles)):
                ax.text(j, i, f"{cam[i,j]:.2f}", ha="center", va="center",
                        color="white" if cam[i,j] < 0.5 else "black", fontsize=6)
        plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()
    plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor=DARK)
    plt.close()
    print(f"  saved → {out_path}")
 
 
def plot_sensitivity_manifold(
    clf_noaug: MLPClassifier,
    Z_by_angle: dict,
    labels: torch.Tensor,
    angles: list,
    n_samples: int = 8,
    out_path: str = ""
):
    """
    For a small sample of test images, visualise the softmax confidence
    profile across all 12 rotations.  Each subplot = one sample; coloured
    lines = per-class probability.
    """
    rng = np.random.default_rng(0)
    idx = rng.choice(len(labels), n_samples, replace=False)
 
    n_classes = clf_noaug(Z_by_angle[0][0:1].to(device)).shape[1]
    cmap      = plt.cm.tab10(np.linspace(0, 1, n_classes))
 
    fig, axes = plt.subplots(2, n_samples // 2, figsize=(20, 8), facecolor=DARK)
    fig.suptitle("Rotation Sensitivity Manifold (MLP-NoAug)\n"
                 "Each panel: softmax probability across all 12 rotations for one test image",
                 color="white", fontsize=11, fontweight="bold")
    axes = axes.flatten()
 
    for panel, s_idx in enumerate(idx):
        ax = axes[panel]
        ax.set_facecolor(PANEL)
        for s in ax.spines.values(): s.set_edgecolor("#444")
        ax.tick_params(colors="white")
        prob_curves = []
        with torch.no_grad():
            for angle in angles:
                z  = Z_by_angle[angle][s_idx:s_idx+1].to(device)
                pr = F.softmax(clf_noaug(z), dim=1).cpu().numpy()[0]
                prob_curves.append(pr)
        prob_curves = np.array(prob_curves)  # (K, C)
        for c in range(n_classes):
            ax.plot(range(len(angles)), prob_curves[:, c],
                    color=cmap[c], lw=1.5, alpha=0.85)
        true_cls = labels[s_idx].item()
        ax.axhline(0.5, color="white", ls="--", lw=0.8, alpha=0.4)
        ax.set_xticks(range(len(angles)))
        ax.set_xticklabels([f"{a}°" for a in angles], fontsize=5, color="white", rotation=45)
        ax.set_ylim(-0.05, 1.05)
        ax.set_title(f"True: {true_cls}", color="white", fontsize=8)
 
    plt.tight_layout()
    plt.show()
    plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor=DARK)
    plt.close()
    print(f"  saved → {out_path}")

In [9]:
def main():
    args = get_args()
    os.makedirs(args.out_dir, exist_ok=True)
    print(f"\n{'='*60}")
    print(f"  Bonus Task · Exp 1: Rotation Sensitivity Baselines")
    print(f"  device = {device}")
    print(f"  output → {args.out_dir}")
    print(f"{'='*60}\n")
 
    # ── Load VAE ──────────────────────────────────────────────────────────────
    vae = load_vae(args.task1_ckpt, args.latent_dim)
 
    # ── Load data ─────────────────────────────────────────────────────────────
    train_imgs, train_lbls, train_angs = load_h5(
        os.path.join(args.data_dir, "rotated_mnist_train.h5"))
    test_imgs,  test_lbls,  test_angs  = load_h5(
        os.path.join(args.data_dir, "rotated_mnist_test.h5"))
 
    ANGLES    = sorted(train_angs.unique().tolist())
    DIGITS    = sorted(train_lbls.unique().tolist())
    N_CLASSES = len(DIGITS)
    # Map original digit labels to contiguous [0, n_classes)
    dig2cls   = {d: i for i, d in enumerate(DIGITS)}
    tr_cls    = torch.tensor([dig2cls[l.item()] for l in train_lbls])
    te_cls    = torch.tensor([dig2cls[l.item()] for l in test_lbls])
 
    print(f"\n  Digits : {DIGITS}")
    print(f"  Angles : {ANGLES}")
 
    # ── Pre-encode latents ────────────────────────────────────────────────────
    print("\n[1] Pre-encoding latents …")
 
    # Training set: NoAug uses only 0° samples; FullAug uses all rotations
    mask_0   = train_angs == 0
    Z_tr_0   = pre_encode(vae, train_imgs[mask_0])
    L_tr_0   = tr_cls[mask_0]
 
    # FullAug training: all angles
    Z_tr_all = pre_encode(vae, train_imgs)
    L_tr_all = tr_cls
 
    # Test set: encode 0° samples, then rotate to get Z_by_angle for eval
    te_mask_0 = test_angs == 0
    te_imgs_0 = test_imgs[te_mask_0]
    te_lbls_0 = te_cls[te_mask_0]
    print(f"  test (0° only): {len(te_imgs_0)} samples")
 
    print("\n[2] Building Z_by_angle for evaluation (rotating 0° test set) …")
    Z_by_angle = pre_encode_all_angles(vae, te_imgs_0, ANGLES)
    print(f"  done: {len(ANGLES)} angle slices × {len(te_imgs_0)} samples")
 
    # ── Train classifiers ─────────────────────────────────────────────────────
    print("\n[3] Training classifiers …")
    clf_noaug,  hist_noaug  = train_classifier(
        Z_tr_0, L_tr_0, args.latent_dim, N_CLASSES,
        args.epochs, args.batch_size, args.lr, tag="MLP-NoAug")
    clf_fullaug, hist_fullaug = train_classifier(
        Z_tr_all, L_tr_all, args.latent_dim, N_CLASSES,
        args.epochs, args.batch_size, args.lr, tag="MLP-FullAug")
 
    # ── Evaluate ──────────────────────────────────────────────────────────────
    print("\n[4] Evaluating per-angle accuracy …")
    acc_noaug  = per_angle_accuracy(clf_noaug,  Z_by_angle, te_lbls_0)
    acc_fullaug = per_angle_accuracy(clf_fullaug, Z_by_angle, te_lbls_0)
 
    print("\n[5] Bootstrap confidence intervals (this may take a moment) …")
    ci_noaug   = {}; ci_fullaug = {}
    for angle in ANGLES:
        mn, lo, hi   = bootstrap_ci(clf_noaug,  Z_by_angle[angle], te_lbls_0, args.n_bootstrap)
        ci_noaug[angle]  = (mn, lo, hi)
        mn2, lo2, hi2 = bootstrap_ci(clf_fullaug, Z_by_angle[angle], te_lbls_0, args.n_bootstrap)
        ci_fullaug[angle] = (mn2, lo2, hi2)
 
    print("\n[6] Rotation Equivariance Error (REE) …")
    ree_noaug  = rotation_equivariance_error(clf_noaug,  Z_by_angle)
    ree_fullaug = rotation_equivariance_error(clf_fullaug, Z_by_angle)
 
    print("\n[7] Confusion-Angle Matrix …")
    cam_noaug  = confusion_angle_matrix(clf_noaug,  Z_by_angle, te_lbls_0, N_CLASSES)
    cam_fullaug = confusion_angle_matrix(clf_fullaug, Z_by_angle, te_lbls_0, N_CLASSES)
 
    # ── Summary ───────────────────────────────────────────────────────────────
    rag_na = rotation_accuracy_gap(acc_noaug)
    rag_fa = rotation_accuracy_gap(acc_fullaug)
 
    print(f"\n{'='*60}")
    print(f"  RESULTS SUMMARY — Exp 1")
    print(f"{'='*60}")
    print(f"\n  MLP-NoAug  (trained 0° only):")
    print(f"    acc(0°)              = {acc_noaug[0]:.4f}")
    print(f"    mean acc (all θ)     = {np.mean(list(acc_noaug.values())):.4f}")
    print(f"    Rotation Acc Gap     = {rag_na:.4f}  ← target for invariant methods")
    print(f"    mean REE             = {np.mean([v for k,v in ree_noaug.items() if k!=0]):.4f}")
 
    print(f"\n  MLP-FullAug (oracle upper bound):")
    print(f"    acc(0°)              = {acc_fullaug[0]:.4f}")
    print(f"    mean acc (all θ)     = {np.mean(list(acc_fullaug.values())):.4f}")
    print(f"    Rotation Acc Gap     = {rag_fa:.4f}  ← best achievable with augmentation")
    print(f"    mean REE             = {np.mean([v for k,v in ree_fullaug.items() if k!=0]):.4f}")
 
    print(f"\n  Invariance headroom = {rag_na - rag_fa:.4f}  "
          f"(exp2–4 should close this)")
 
    # ── Save artefacts ────────────────────────────────────────────────────────
    print("\n[8] Saving checkpoints and figures …")
    torch.save({"noaug":  clf_noaug.state_dict(),
                "fullaug": clf_fullaug.state_dict()},
               os.path.join(args.out_dir, "classifiers.pt"))
 
    np.savez(os.path.join(args.out_dir, "baseline_results.npz"),
             angles=ANGLES,
             acc_noaug   =[acc_noaug[a]  for a in ANGLES],
             acc_fullaug =[acc_fullaug[a] for a in ANGLES],
             ree_noaug   =[ree_noaug[a]  for a in ANGLES],
             ree_fullaug =[ree_fullaug[a] for a in ANGLES],
             cam_noaug   =cam_noaug,
             cam_fullaug =cam_fullaug,
             rag_noaug   =rag_na,
             rag_fullaug =rag_fa,
             n_classes   =N_CLASSES,
             digits      =DIGITS)
 
    print("\n[9] Generating figures …")
    plot_accuracy_profile(
        acc_noaug, acc_fullaug,
        ci_noaug, ci_fullaug,
        ree_noaug, ree_fullaug,
        hist_noaug, hist_fullaug,
        os.path.join(args.out_dir, "fig_accuracy_profile.png"))
 
    plot_confusion_angle(
        cam_noaug, cam_fullaug, ANGLES, DIGITS,
        os.path.join(args.out_dir, "fig_confusion_angle.png"))
 
    plot_sensitivity_manifold(
        clf_noaug, Z_by_angle, te_lbls_0, ANGLES,
        n_samples=8,
        out_path=os.path.join(args.out_dir, "fig_sensitivity_manifold.png"))
 
    print(f"\n✅  Exp 1 complete.  Outputs → {args.out_dir}/")
    print(f"    Key takeaway: RAG(NoAug) = {rag_na:.4f}" )

In [10]:
if __name__ == "__main__":
    main()


  Bonus Task · Exp 1: Rotation Sensitivity Baselines
  device = cuda
  output → ./bonus_outputs/exp1

  ✓ Task-1 VAE loaded  (epoch 47)
  loading rotated_mnist_train.h5 ... done (2.7s)  shape=(152400, 1, 28, 28)
  loading rotated_mnist_test.h5 ... done (0.5s)  shape=(26004, 1, 28, 28)

  Digits : [1, 2]
  Angles : [0, 30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330]

[1] Pre-encoding latents …


  encoding:   0%|          | 0/25 [00:00<?, ?it/s]

  encoding:   0%|          | 0/298 [00:00<?, ?it/s]

  test (0° only): 2167 samples

[2] Building Z_by_angle for evaluation (rotating 0° test set) …
  done: 12 angle slices × 2167 samples

[3] Training classifiers …

  Training MLP-NoAug for 30 epochs …
    ep   1/30  acc=0.9788  loss=0.0782
    ep  10/30  acc=0.9917  loss=0.0283
    ep  20/30  acc=0.9926  loss=0.0252
    ep  30/30  acc=0.9928  loss=0.0226

  Training MLP-FullAug for 30 epochs …
    ep   1/30  acc=0.9878  loss=0.0535
    ep  10/30  acc=0.9889  loss=0.0377
    ep  20/30  acc=0.9899  loss=0.0320
    ep  30/30  acc=0.9907  loss=0.0281

[4] Evaluating per-angle accuracy …

[5] Bootstrap confidence intervals (this may take a moment) …

[6] Rotation Equivariance Error (REE) …

[7] Confusion-Angle Matrix …

  RESULTS SUMMARY — Exp 1

  MLP-NoAug  (trained 0° only):
    acc(0°)              = 0.9972
    mean acc (all θ)     = 0.8496
    Rotation Acc Gap     = 0.1611  ← target for invariant methods
    mean REE             = 0.2387

  MLP-FullAug (oracle upper bound):
    acc(0°)